# Questão 3 - CUSTOS_IMPORTACAO

Esta etapa tem como objetivo estruturar os dados do arquivo custos_importacao.json, que possuem informações aninhadas de preços, transformando-os em um formato tabular para geração de um arquivo CSV adequado para análise. Para isso, foi utilizada a linguagem Python 3 em conjunto com o ambiente Snowflake.

# Parte 1
O código realiza a conexão com o Snowflake utilizando Snowpark e credenciais seguras via dotenv. Em seguida, a tabela custos_importacao é carregada e convertida para um DataFrame pandas, permitindo a manipulação dos dados.

Essa etapa prepara a base para o tratamento do JSON e a posterior geração do arquivo CSV estrutur

In [ ]:
import pandas as pd
import os
import json
from dotenv import load_dotenv
from snowflake.snowpark import Session

load_dotenv()

conn = {
    "account": os.getenv("account"),
    "user": os.getenv("user"),
    "password": os.getenv("password"),
    "role": "ACCOUNTADMIN",
    "warehouse": "COMPUTE_WH",
    "database": "LH_NAUTICALS"
}

session = Session.builder.configs(conn).create()


df_snowpark = session.table("LH_NAUTICALS.QUESTAO3.custos_importacao")


df_pandas = df_snowpark.to_pandas()


## Tratamento do JSON e Geração do CSV

Foi realizado o tratamento dos dados aninhados da coluna variant_col, convertendo o conteúdo JSON em uma estrutura tabular.

O código percorre os registros, extrai informações dos produtos e seus históricos de preço, e organiza os dados em um novo DataFrame. Em seguida, realiza a conversão de tipos (data e valor numérico) para garantir consistência.

Por fim, os dados tratados são exportados para um arquivo CSV, resultando em uma base estruturada e pronta para análise.

In [ ]:

print("Colunas disponíveis:", df_pandas.columns.tolist())
print(df_pandas.head())


linhas_tratadas = []


for _, row in df_pandas.iterrows():
    dados = row["variant_col"]   

    
    if isinstance(dados, str):
        dados = json.loads(dados)

    
    if not isinstance(dados, dict):
        continue

    
    product_id = dados.get("product_id")
    product_name = dados.get("product_name")
    category = dados.get("category")
    historic_data = dados.get("historic_data", [])

    
    if not isinstance(historic_data, list):
        continue

    
    for historico in historic_data:
        if isinstance(historico, dict):
            linhas_tratadas.append({
                "product_id": product_id,
                "product_name": product_name,
                "category": category,
                "start_date": historico.get("start_date"),
                "usd_price": historico.get("usd_price")
            })

df_custos_importacao = pd.DataFrame(linhas_tratadas)

df_custos_importacao["start_date"] = pd.to_datetime(
    df_custos_importacao["start_date"],
    dayfirst=True,
    errors="coerce"
)

df_custos_importacao["usd_price"] = pd.to_numeric(
    df_custos_importacao["usd_price"],
    errors="coerce"
)


print(df_custos_importacao.head())
print(df_custos_importacao.info())


df_custos_importacao.to_csv(
    "custos_importacao_tratado.csv",
    index=False,
    encoding="utf-8"
)

print("Arquivo CSV gerado com sucesso.")

Colunas disponíveis: ['variant_col']
                                         variant_col
0  {\n  "category": "eletrônicos",\n  "historic_d...
1  {\n  "category": "eletrônicos",\n  "historic_d...
2  {\n  "category": "eletrônicos",\n  "historic_d...
3  {\n  "category": "eletrônicos",\n  "historic_d...
4  {\n  "category": "eletrônicos",\n  "historic_d...
   product_id                 product_name     category start_date  usd_price
0           1  Transponder AIS Maré Magnum  eletrônicos 2016-08-10   10583.63
1           1  Transponder AIS Maré Magnum  eletrônicos 2018-06-15    8778.36
2           1  Transponder AIS Maré Magnum  eletrônicos 2018-09-25    8023.87
3           1  Transponder AIS Maré Magnum  eletrônicos 2019-03-19    8772.78
4           1  Transponder AIS Maré Magnum  eletrônicos 2020-01-17    7918.18
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1260 entries, 0 to 1259
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --

# Validação
O comando calcula e exibe a quantidade total de entradas presentes no DataFrame df_custos_importacao, permitindo validar o volume de dados gerado após o processo de transformação.

In [8]:
print("Total de entradas:", len(df_custos_importacao))


Total de entradas: 1260


## Conclusão

O processo realizado permitiu transformar dados originalmente aninhados em formato JSON em uma estrutura tabular organizada, adequada para armazenamento e análise.

Por meio das etapas de extração, tratamento, validação e conversão de tipos, foi possível garantir a consistência, integridade e qualidade dos dados. A geração do arquivo CSV final facilita consultas, análises e integração com outras ferramentas.

Dessa forma, a solução atende aos requisitos propostos, estruturando corretamente os dados históricos de preços e tornando a base pronta para uso em análises e tomada de decisão.